# 🛰️ Orbital RL — Quick Start Demo

This notebook demonstrates the orbital simulation environment, shows how the CW physics works, 
and runs a trained agent (if available).

```bash
pip install -r requirements.txt
```

In [ ]:
import sys
sys.path.insert(0, '..')  # so we can import envs from the notebooks/ folder

import numpy as np
import matplotlib.pyplot as plt
from envs.orbital_env import OrbitalEnv
from envs.dynamics import mean_motion, propagate

print('✓ Imports OK')

## 1. Orbital Parameters

We simulate at **400 km altitude** (Low Earth Orbit), like the ISS.

In [ ]:
n = mean_motion(400.0)
T_orbit = 2 * np.pi / n

print(f'Mean motion n = {n:.6e} rad/s')
print(f'Orbital period = {T_orbit:.0f} s  ({T_orbit/60:.1f} min)')

## 2. Free Drift Trajectory

A spacecraft displaced 500 m radially with no thrust — traces a natural CW ellipse.

In [ ]:
# Simulate free drift
s = np.array([500.0, 0.0, 0.0, 0.0])  # 500 m radial offset
trajectory = [s.copy()]

for _ in range(int(T_orbit)):
    s = propagate(s, n, np.zeros(2), dt=1.0)
    trajectory.append(s.copy())

traj = np.array(trajectory)

fig, ax = plt.subplots(figsize=(7, 7), facecolor='#0a0a12')
ax.set_facecolor('#0a0a12')
ax.plot(traj[:, 1], traj[:, 0], color='#00d4ff', lw=1.2, label='Free drift')
ax.plot(traj[0, 1], traj[0, 0], 'o', color='lime', ms=10, label='Start', zorder=5)
ax.plot(0, 0, 'x', color='#ffd700', ms=14, mew=3, label='Target station')
ax.set_xlabel('Along-track y [m]', color='white')
ax.set_ylabel('Radial x [m]', color='white')
ax.set_title('CW Free Drift — 1 Full Orbit', color='white', fontsize=12)
ax.tick_params(colors='#666')
ax.legend(facecolor='#111', labelcolor='white')
ax.spines[:].set_color('#333')
plt.tight_layout()
plt.show()

print(f'Start position:  x={trajectory[0][0]:.1f} m, y={trajectory[0][1]:.1f} m')
print(f'End position:    x={trajectory[-1][0]:.1f} m, y={trajectory[-1][1]:.1f} m')

## 3. Gymnasium Environment — Random Agent

Let's create the docking environment and run a random policy to see what happens 
(spoiler: it's terrible — that's why we need RL).

In [ ]:
env = OrbitalEnv(task='docking', max_steps=500)
obs, _ = env.reset(seed=42)

print('Observation space:', env.observation_space)
print('Action space:', env.action_space)
print(f'Initial observation: {obs}')
print(f'Initial state: x={env._state[0]:.1f}m, y={env._state[1]:.1f}m')

In [ ]:
# Run random agent for one episode
obs, _ = env.reset(seed=42)
total_reward = 0.0
done = truncated = False

while not (done or truncated):
    action = env.action_space.sample()
    obs, reward, done, truncated, info = env.step(action)
    total_reward += reward

print(f'\nEpisode finished:')
print(f'  Outcome:  {info.get("outcome", "unknown")}')
print(f'  Steps:    {info["steps"]}')
print(f'  Distance: {info["distance"]:.1f} m')
print(f'  Speed:    {info["speed"]:.2f} m/s')
print(f'  Fuel:     {info["fuel"]:.1f} kg')
print(f'  Reward:   {total_reward:.1f}')

In [ ]:
# Plot the random agent's trajectory
traj = np.array(env._trajectory)

fig, ax = plt.subplots(figsize=(7, 7), facecolor='#0a0a12')
ax.set_facecolor('#0a0a12')
ax.plot(traj[:, 1], traj[:, 0], '-', color='#ff6ec7', lw=0.8, alpha=0.7, label='Random agent')
ax.plot(traj[0, 1], traj[0, 0], 'o', color='lime', ms=10, label='Start', zorder=5)
ax.plot(traj[-1, 1], traj[-1, 0], 's', color='red', ms=8, label='End', zorder=5)
ax.plot(0, 0, 'x', color='#ffd700', ms=14, mew=3, label='Target')

# Draw dock radius
circle = plt.Circle((0, 0), env.dock_radius, color='lime', fill=False, lw=0.5, ls='--', label='Dock radius')
ax.add_patch(circle)

ax.set_xlabel('Along-track y [m]', color='white')
ax.set_ylabel('Radial x [m]', color='white')
ax.set_title(f'Random Agent Trajectory — {info.get("outcome", "")}', color='white', fontsize=12)
ax.tick_params(colors='#666')
ax.legend(facecolor='#111', labelcolor='white')
ax.spines[:].set_color('#333')
plt.tight_layout()
plt.show()

## 4. Load & Evaluate a Trained Agent

After training (`python train.py --task docking`), uncomment the cell below to watch the PPO agent dock.

In [ ]:
# # Uncomment after training:
# from stable_baselines3 import PPO
# model = PPO.load('../models/docking/best/best_model')
#
# env = OrbitalEnv(task='docking', max_steps=1000)
# obs, _ = env.reset(seed=123)
# done = truncated = False
#
# while not (done or truncated):
#     action, _ = model.predict(obs, deterministic=True)
#     obs, reward, done, truncated, info = env.step(action)
#
# traj = np.array(env._trajectory)
# fig, ax = plt.subplots(figsize=(7, 7), facecolor='#0a0a12')
# ax.set_facecolor('#0a0a12')
# ax.plot(traj[:, 1], traj[:, 0], '-', color='#00d4ff', lw=1.0)
# ax.plot(traj[0, 1], traj[0, 0], 'o', color='lime', ms=10, label='Start')
# ax.plot(traj[-1, 1], traj[-1, 0], 's', color='cyan', ms=8, label='End')
# ax.plot(0, 0, 'x', color='#ffd700', ms=14, mew=3, label='Target')
# ax.set_title(f'PPO Agent — {info.get("outcome", "")}', color='white')
# ax.legend(facecolor='#111', labelcolor='white')
# plt.show()
# print(f'Outcome: {info["outcome"]} | Distance: {info["distance"]:.2f}m | Fuel: {info["fuel"]:.1f}kg')